# IVA Gradient Artifact Removal Demo

Este notebook carga un registro simultaneo EEG-fMRI (`fmrirestingec`), aplica limpieza del artefacto de gradiente con un pipeline IVA-GL (`IVA-G` seguido de `IVA-L-SOS`), muestra la senal antes y despues de la limpieza, y ofrece una etapa intermedia para inspeccionar las SCVs extraidas con visualizacion topologica estilo MNE.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault("_MNE_FAKE_HOME_DIR", str(PROJECT_ROOT / ".mne_local"))
(PROJECT_ROOT / ".mne_local").mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import mne
import numpy as np

from functions.iva_ga import (
    apply_iva_ga_to_raw,
    plot_iva_component_interactive,
    plot_iva_component_topomaps,
)


In [15]:
DEFAULT_EEG_ROOT = Path(
    r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG"
)
SUBJECT = "sub-001"
TASK = "fmrirestingec"
TR = 2.0
OFFSET = 0
PERIODIC_FREQUENCY_HZ = 14.0
EXCLUDE_LAST_CHANNEL = True
DISPLAY_SECONDS = 20
PROCESS_DURATION_SECONDS = 300
IVA_G_MAX_ITER = 64
IVA_L_MAX_ITER = 128
RANDOM_STATE = 0


El ajuste `PROCESS_DURATION_SECONDS` deja el ejemplo en una ventana corta por defecto, porque en esta formulacion directa `N = n_epochs` y el coste computacional crece rapido. Si quieres aplicar el metodo al registro completo, cambia ese valor a `None`.

In [16]:
def get_eeg_set_path(subject: str, task: str = TASK, eeg_root: Path = DEFAULT_EEG_ROOT) -> Path:
    eeg_path = eeg_root / subject / "eeg" / f"{subject}_task-{task}_eeg.set"
    if not eeg_path.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_path}")
    return eeg_path


def load_raw_eeg(eeg_path: Path) -> mne.io.BaseRaw:
    return mne.io.read_raw_eeglab(eeg_path, preload=True, verbose="ERROR")


def prepare_raw_for_iva(raw: mne.io.BaseRaw, exclude_last_channel: bool = True) -> mne.io.BaseRaw:
    if exclude_last_channel:
        return raw.copy().pick(raw.ch_names[:-1])
    return raw.copy()


def maybe_crop_raw(raw: mne.io.BaseRaw, max_seconds: float | None) -> mne.io.BaseRaw:
    if max_seconds is None:
        return raw.copy()
    return raw.copy().crop(tmin=0.0, tmax=min(float(max_seconds), raw.times[-1]))


def plot_channel_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    n_samples = min(int(round(duration_s * fs)), raw_before.n_times, raw_after.n_times)
    time = np.arange(n_samples) / fs
    before = raw_before.get_data(picks=[channel_index])[0, :n_samples]
    after = raw_after.get_data(picks=[channel_index])[0, :n_samples]
    channel_name = raw_before.ch_names[channel_index]

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axes[0].plot(time, before, linewidth=0.8)
    axes[0].set_title(f"Before IVA-GA | {channel_name}")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time, after, linewidth=0.8, color="tab:orange")
    axes[1].set_title(f"After IVA-GA | {channel_name}")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Amplitude")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_overlay_before_after(
    raw_before: mne.io.BaseRaw,
    raw_after: mne.io.BaseRaw,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    n_samples = min(int(round(duration_s * fs)), raw_before.n_times, raw_after.n_times)
    time = np.arange(n_samples) / fs
    before = raw_before.get_data(picks=[channel_index])[0, :n_samples]
    after = raw_after.get_data(picks=[channel_index])[0, :n_samples]
    channel_name = raw_before.ch_names[channel_index]

    plt.figure(figsize=(14, 4))
    plt.plot(time, before, label="Before IVA-GA", linewidth=0.8, alpha=0.75)
    plt.plot(time, after, label="After IVA-GA", linewidth=0.8, alpha=0.75)
    plt.title(f"Overlay Before/After | {channel_name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [17]:
eeg_path = get_eeg_set_path(SUBJECT)
raw = load_raw_eeg(eeg_path)
raw_iva = prepare_raw_for_iva(raw, exclude_last_channel=EXCLUDE_LAST_CHANNEL)
raw_iva = maybe_crop_raw(raw_iva, PROCESS_DURATION_SECONDS)
fs = float(raw_iva.info["sfreq"])

print(f"Subject: {SUBJECT}")
print(f"Task: {TASK}")
print(f"EEG path: {eeg_path}")
print(f"Original shape: {raw.get_data().shape}")
print(f"IVA input shape: {raw_iva.get_data().shape}")
print(f"Sampling frequency: {fs} Hz")
print(f"TR: {TR} s")
print(f"Process duration: {PROCESS_DURATION_SECONDS} s")
if EXCLUDE_LAST_CHANNEL:
    print(f"Excluded channel: {raw.ch_names[-1]}")


Subject: sub-001
Task: fmrirestingec
EEG path: C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-001\eeg\sub-001_task-fmrirestingec_eeg.set
Original shape: (33, 639320)
IVA input shape: (32, 300001)
Sampling frequency: 1000.0 Hz
TR: 2.0 s
Process duration: 300 s
Excluded channel: ECG


## Visualizacion interactiva del EEG original

In [18]:
raw_iva.plot(duration=min(15, raw_iva.times[-1]), scalings="auto", n_channels=min(20, len(raw_iva.ch_names)))

## Aplicacion del pipeline IVA-GA

In [19]:
raw_clean, result = apply_iva_ga_to_raw(
    raw_iva,
    TR=TR,
    exclude_last_channel=False,
    offset=OFFSET,
    periodic_frequency_hz=PERIODIC_FREQUENCY_HZ,
    iva_g_max_iter=IVA_G_MAX_ITER,
    iva_l_max_iter=IVA_L_MAX_ITER,
    whiten=True,
    verbose=False,
    random_state=RANDOM_STATE,
)

print(f"Library used: {result['library_used']}")
print(f"Samples per TR: {result['T_samples']}")
print(f"Detected GA component: {result['ga_component']}")
if 'warning' in result:
    print(f"Warning: {result['warning']}")

for idx in range(min(10, len(result['combined_scores']))):
    print(
        f"SCV {idx:02d} | combined={result['combined_scores'][idx]:.4f} | "
        f"dependence={result['dependence_scores'][idx]:.4f} | "
        f"mi={result['mi_scores'][idx]:.4f} | harmonic={result['harmonic_scores'][idx]:.4f}"
    )


Library used: independent_vector_analysis 0.3.6
Samples per TR: 2000
Detected GA component: 31
SCV 00 | combined=0.2387 | dependence=0.5256 | mi=0.4979 | harmonic=0.0011
SCV 01 | combined=0.4741 | dependence=0.5046 | mi=1.4893 | harmonic=0.0020
SCV 02 | combined=0.1947 | dependence=0.5241 | mi=0.2750 | harmonic=0.0170
SCV 03 | combined=0.4682 | dependence=0.5371 | mi=1.3919 | harmonic=0.0019
SCV 04 | combined=0.2247 | dependence=0.5189 | mi=0.4069 | harmonic=0.0171
SCV 05 | combined=0.3478 | dependence=0.8323 | mi=0.1908 | harmonic=0.0161
SCV 06 | combined=0.3414 | dependence=0.8263 | mi=0.1872 | harmonic=0.0134
SCV 07 | combined=0.3392 | dependence=0.8196 | mi=0.1940 | harmonic=0.0132
SCV 08 | combined=0.3200 | dependence=0.7889 | mi=0.1901 | harmonic=0.0122
SCV 09 | combined=0.3416 | dependence=0.8211 | mi=0.1922 | harmonic=0.0157


## Exploracion interactiva de SCVs

In [ ]:
plot_iva_component_interactive(raw_iva, result, picks=raw_iva.ch_names)

## Visualizacion interactiva despues de limpiar

In [22]:
raw_clean.plot(duration=min(15, raw_clean.times[-1]), scalings="auto", n_channels=min(20, len(raw_clean.ch_names)))

## Comparacion local antes y despues

In [ ]:
plot_channel_before_after(raw_iva, raw_clean, channel_index=0, duration_s=DISPLAY_SECONDS)
plot_overlay_before_after(raw_iva, raw_clean, channel_index=0, duration_s=DISPLAY_SECONDS)